# 05 - Multimodal Fusion

Combines the vision and genomics branches. Since PCam/BreakHis and ClinVar are not natively patient-matched, this notebook builds a **simulated paired dataset** (randomly pairing an image sample with a variant sample and assigning a joint label) purely to demonstrate the fusion architecture end-to-end. Replace with real patient-matched data (e.g. TCGA) for a clinically meaningful version — see `docs/future_work.md`.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import torch
from src.utils.helpers import load_config
from src.utils.seed import set_seed

cfg = load_config('../config/config.yaml')
set_seed(cfg['seed'])

## Load trained branch models (run 03 and 04 first, or the CLI training scripts)

In [ ]:
from src.inference import load_vision_model, load_genomics_model

vision_model = load_vision_model('../models/vision_best.pt')
genomics_model = load_genomics_model()
print('vision model loaded:', vision_model is not None)
print('genomics model loaded:', genomics_model is not None)

## Build the fusion classifier

In [ ]:
from src.multimodal.fusion import LateFusionClassifier

if vision_model is not None and genomics_model is not None:
    fusion_model = LateFusionClassifier(
        vision_dim=vision_model.embed_dim,
        genomics_dim=genomics_model.embed_dim,
        hidden_dim=cfg['multimodal']['hidden_dim'],
    )
    print(fusion_model)
else:
    print('Train both branch models first (see notebooks 03 and 04).')

## Embedding correlation analysis (training-free diagnostic)

In [ ]:
from src.multimodal.fusion import embedding_correlation

# Example with random embeddings as a placeholder — replace with real
# extracted embeddings from vision_model.get_embedding() / genomics_model.get_embedding()
# over a paired sample set.
rng = np.random.default_rng(42)
vision_embs = rng.normal(size=(100, 64))
genomics_embs = vision_embs * 0.6 + rng.normal(size=(100, 64)) * 0.8  # correlated toy example
corr, p_value = embedding_correlation(vision_embs, genomics_embs)
print(f'Pearson correlation: {corr:.3f} (p={p_value:.4f})')

## Training loop sketch

For a real paired dataset, train the fusion head with a standard classification loop (see `src/vision/train.py` / `src/genomics/train.py` for the pattern) and save the checkpoint to `models/fusion_best.pt` so `app/pages/3_Multimodal_Demo.py` picks it up automatically.